# Лабораторная работа №1 по интервальному анализу


In [1]:
import itertools
import numpy as np
import pandas as pd

np.set_printoptions(precision=9, suppress=True)
TOL = 1e-10


## Исходные данные


In [2]:
A1_MID = np.array([
    [0.95, 1.00],
    [1.05, 1.00],
    [1.10, 1.00],
], dtype=float)

A1_RAD_TOMO = np.ones((3, 2), dtype=float)
A1_RAD_REGRESSION = np.array([
    [1.0, 0.0],
    [1.0, 0.0],
    [1.0, 0.0],
])

A2_MID = np.array([
    [1.10, 0.90, 1.10],
    [1.40, 1.00, 0.80],
    [0.80, 1.40, 1.20],
], dtype=float)
A2_RAD = np.ones((3, 3), dtype=float)

print('mid(A1) =')
print(A1_MID)
print('\nmid(A2) =')
print(A2_MID)


mid(A1) =
[[0.95 1.  ]
 [1.05 1.  ]
 [1.1  1.  ]]

mid(A2) =
[[1.1 0.9 1.1]
 [1.4 1.  0.8]
 [0.8 1.4 1.2]]


In [3]:
def belongs(A_point, A_mid, Delta, delta, tol=TOL):
    return np.all(np.abs(A_point - A_mid) <= delta * Delta + tol)

def matrix_check(A_point, A_mid, Delta, delta):
    return pd.DataFrame({
        "max |A_prime - mid(A)|": [np.max(np.abs(A_point - A_mid))],
        'max allowed': [np.max(delta * Delta)],
        'belongs': [belongs(A_point, A_mid, Delta, delta)]
    })


## 1. $A_1$: томография

Поиск критического $\delta$ и проверка найденной граничной матрицы.


In [4]:
c = A1_MID[:, 0]
c_min, c_max = c.min(), c.max()
lam_tomo = (c_max + c_min) / 2
delta_tomo = (c_max - c_min) / (c_max + c_min + 2 * A1_MID[0, 1])

col2 = np.where(
    c == c_max, 1 + delta_tomo,
    np.where(c == c_min, 1 - delta_tomo, 1.0)
)
A1_tomo = np.column_stack([lam_tomo * col2, col2])

print(f'delta* = {delta_tomo:.12f}')
print(f'lambda = {lam_tomo:.12f}')
print('A1_prime =')
print(A1_tomo)
print('rank(A1_prime) =', np.linalg.matrix_rank(A1_tomo, tol=1e-9))
print('first - lambda*second =', A1_tomo[:, 0] - lam_tomo * A1_tomo[:, 1])
print('belongs =', belongs(A1_tomo, A1_MID, A1_RAD_TOMO, delta_tomo))


delta* = 0.037037037037
lambda = 1.025000000000
A1_prime =
[[0.987037037 0.962962963]
 [1.025       1.         ]
 [1.062962963 1.037037037]]
rank(A1_prime) = 1
first - lambda*second = [0. 0. 0.]
belongs = True


In [5]:
minors_tomo = []
for i, j in itertools.combinations(range(3), 2):
    minors_tomo.append(np.linalg.det(A1_tomo[[i, j], :]))

print('Миноры 2x2:', np.array(minors_tomo))
print('|A1_prime - mid(A1)| =')
print(np.abs(A1_tomo - A1_MID))
print('delta*Delta =')
print(delta_tomo * A1_RAD_TOMO)


Миноры 2x2: [ 0. -0. -0.]
|A1_prime - mid(A1)| =
[[0.037037037 0.037037037]
 [0.025       0.         ]
 [0.037037037 0.037037037]]
delta*Delta =
[[0.037037037 0.037037037]
 [0.037037037 0.037037037]
 [0.037037037 0.037037037]]


## 2. $A_1$: регрессия

Поиск критического $\delta$ и проверка найденной граничной матрицы.


In [6]:
delta_reg = (A1_MID[:,0].max() - A1_MID[:,0].min()) / 2
lam_reg = (A1_MID[:,0].max() + A1_MID[:,0].min()) / 2
A1_reg = np.column_stack([np.full(3, lam_reg), A1_MID[:,1]])

print(f'delta* = {delta_reg:.12f}')
print('A1_prime =')
print(A1_reg)
print('rank(A1_prime) =' , np.linalg.matrix_rank(A1_reg, tol=1e-9))
print('belongs =', belongs(A1_reg, A1_MID, A1_RAD_REGRESSION, delta_reg))
print('|A1_prime - mid(A1)| =')
print(np.abs(A1_reg - A1_MID))
print('delta*Delta =')
print(delta_reg * A1_RAD_REGRESSION)


delta* = 0.075000000000
A1_prime =
[[1.025 1.   ]
 [1.025 1.   ]
 [1.025 1.   ]]
rank(A1_prime) = 1
belongs = True
|A1_prime - mid(A1)| =
[[0.075 0.   ]
 [0.025 0.   ]
 [0.075 0.   ]]
delta*Delta =
[[0.075 0.   ]
 [0.075 0.   ]
 [0.075 0.   ]]


## 3. $A_2$: квадратная матрица

Поиск критического $\delta$, построение особенной матрицы и проверка результата.


In [7]:
A2_inv = np.linalg.inv(A2_MID)
rows = []
best = None

for signs in itertools.product([-1.0, 1.0], repeat=3):
    y = np.array(signs)
    v = A2_inv @ y
    norm1 = np.linalg.norm(v, 1)
    rows.append([str(tuple(map(int, y))), *v, norm1])
    if best is None or norm1 > best[0] + TOL:
        best = (norm1, y, v)

sign_table = pd.DataFrame(rows, columns=['y', '(A^-1 y)_1', '(A^-1 y)_2', '(A^-1 y)_3', 'L1 norm'])
sign_table


,y,(A^-1 y)_1,(A^-1 y)_2,(A^-1 y)_3,L1 norm
0,"(-1, -1, -1)",-3.738318e-01,-0.140187,-0.420561,0.934579
1,"(-1, -1, 1)",-2.149533e+00,2.943925,-1.168224,6.261682
2,"(-1, 1, -1)",1.775701e+00,1.915888,-4.252336,7.943925
3,"(-1, 1, 1)",3.330669e-16,5.000000,-5.000000,10.000000
4,"(1, -1, -1)",-3.330669e-16,-5.000000,5.000000,10.000000
5,"(1, -1, 1)",-1.775701e+00,-1.915888,4.252336,7.943925
6,"(1, 1, -1)",2.149533e+00,-2.943925,1.168224,6.261682
7,"(1, 1, 1)",3.738318e-01,0.140187,0.420561,0.934579


In [8]:
best_norm, best_y, best_v = best
delta_a2 = 1 / best_norm
# При нулевой компоненте знак можно выбрать произвольно; берём +1.
z = np.where(best_v >= 0, 1.0, -1.0)
A2_point = A2_MID - delta_a2 * np.outer(best_y, z)

print('best y =', best_y.astype(int))
print('A^-1 y =', best_v)
print('z =', z.astype(int))
print(f'delta* = {delta_a2:.12f}')
print('A2_prime =')
print(A2_point)
print(f'det(A2_prime) = {np.linalg.det(A2_point):.16e}')
print('rank(A2_prime) =' , np.linalg.matrix_rank(A2_point, tol=1e-9))
print('A2_prime @ (A^-1 y) =' , A2_point @ best_v)
print('belongs =', belongs(A2_point, A2_MID, A2_RAD, delta_a2))


best y = [-1  1  1]
A^-1 y = [ 0.  5. -5.]
z = [ 1  1 -1]
delta* = 0.100000000000
A2_prime =
[[1.2 1.  1. ]
 [1.3 0.9 0.9]
 [0.7 1.3 1.3]]
det(A2_prime) = 0.0000000000000000e+00
rank(A2_prime) = 2
A2_prime @ (A^-1 y) = [ 0. -0. -0.]
belongs = True


In [9]:
print('|A2_prime - mid(A2)| =')
print(np.abs(A2_point - A2_MID))
print('delta*Delta =')
print(delta_a2 * A2_RAD)


|A2_prime - mid(A2)| =
[[0.1 0.1 0.1]
 [0.1 0.1 0.1]
 [0.1 0.1 0.1]]
delta*Delta =
[[0.1 0.1 0.1]
 [0.1 0.1 0.1]
 [0.1 0.1 0.1]]


### Дополнительная проверка $A_2$

Контроль результата полным перебором 512 знаковых направлений.


In [10]:
def brute_force_square_radius(Ac, tol=1e-9):
    xs = np.array([0.0, 1.0, -1.0, 2.0])
    V = np.vander(xs, 4, increasing=True)  # коэффициенты c0..c3

    best_delta = np.inf
    best_S = None

    for bits in itertools.product([1.0, -1.0], repeat=Ac.size):
        S = np.array(bits).reshape(Ac.shape)
        ys = np.array([np.linalg.det(Ac + x * S) for x in xs])
        coeff = np.linalg.solve(V, ys)
        roots = np.roots(coeff[::-1])

        for root in roots:
            if abs(root.imag) < tol and root.real > tol and root.real < best_delta:
                best_delta = root.real
                best_S = S.copy()

    return best_delta, best_S

brute_delta, brute_S = brute_force_square_radius(A2_MID)
print(f'Перебор 512 направлений: delta* = {brute_delta:.12f}')
print(f'Основной метод:            delta* = {delta_a2:.12f}')
print(f'Разница = {abs(brute_delta - delta_a2):.3e}')
print('Направление S, на котором достигается граница:')
print(brute_S)
print('det(A2 + delta*S) =', np.linalg.det(A2_MID + brute_delta * brute_S))


Перебор 512 направлений: delta* = 0.100000000000
Основной метод:            delta* = 0.100000000000
Разница = 1.388e-16
Направление S, на котором достигается граница:
[[-1.  1. -1.]
 [ 1. -1.  1.]
 [-1. -1.  1.]]
det(A2 + delta*S) = 8.792966355031251e-16


## Итоговые результаты


In [11]:
summary = pd.DataFrame({
    'Случай': [
        'A1: томография, оба столбца интервальные',
        'A1: регрессия, интервальный только первый столбец',
        'A2: квадратная матрица'
    ],
    'delta*': [delta_tomo, delta_reg, delta_a2],
    'Диапазон появления особенной реализации': [
        f'delta >= {delta_tomo:.9f}',
        f'delta >= {delta_reg:.3f}',
        f'delta >= {delta_a2:.3f}'
    ]
})
summary


,Случай,delta*,Диапазон появления особенной реализации
0,"A1: томография, оба столбца интервальные",0.037037,delta >= 0.037037037
1,"A1: регрессия, интервальный только первый столбец",0.075000,delta >= 0.075
2,A2: квадратная матрица,0.100000,delta >= 0.100
